In [1]:
import pandas as pd

In [2]:
#Loading Data

df = pd.read_csv(r'Cardiology_Medicare_Part_D_Prescribers_by_Provider_2024.csv')

C:\Users\sahil\AppData\Local\Temp\ipykernel_21300\1577287296.py:3: DtypeWarning: Columns (0: Opioid_Tot_Clms, 1: Opioid_LA_Tot_Clms, 2: Antbtc_Tot_Clms, 3: Bene_Age_LT_65_Cnt, 4: Bene_Age_65_74_Cnt, 5: Bene_Age_75_84_Cnt, 6: Bene_Age_GT_84_Cnt, 7: Bene_Race_Black_Cnt, 8: Bene_Race_Othr_Cnt) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'Cardiology_Medicare_Part_D_Prescribers_by_Provider_2024.csv')


In [3]:
df.head(1)

,PRSCRBR_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_MI,Prscrbr_Crdntls,Prscrbr_Ent_Cd,Prscrbr_St1,Prscrbr_St2,Prscrbr_City,Prscrbr_State_Abrvtn,...,Bene_Male_Cnt,Bene_Race_Wht_Cnt,Bene_Race_Black_Cnt,Bene_Race_Api_Cnt,Bene_Race_Hspnc_Cnt,Bene_Race_Natind_Cnt,Bene_Race_Othr_Cnt,Bene_Dual_Cnt,Bene_Ndual_Cnt,Bene_Avg_Risk_Scre
0,1003000936,Stellingworth,Mark,A,MD,I,701 Medical Park Dr Ste 301,NaN,Hartsville,SC,...,118,165,102,NaN,0,0.0,NaN,76,193,1.539


In [4]:
#Creating a lean dataframe with only the required columns

columns_to_keep = [
    'PRSCRBR_NPI',            # Unique Doctor ID (National Provider Identifier)
    'Prscrbr_Last_Org_Name',  # Last Name
    'Prscrbr_First_Name',     # First Name
    'Prscrbr_City',           # City (For mapping territories later)
    'Prscrbr_State_Abrvtn',   # State
    'Tot_Clms',               # Total Prescriptions/Claims (Our primary KPI)
    'Tot_Drug_Cst'            # Total Revenue/Cost
]

df_lean = df[columns_to_keep].copy()


In [5]:
#Dropping rows where critical data is missing (null values) in 'PRSCRBR_NPI' and 'Tot_Clms' columns
#Dropping rows with null values in 'PRSCRBR_NPI' and 'Tot_Clms' columns

df_lean = df_lean.dropna(subset=['PRSCRBR_NPI', 'Tot_Clms'])

In [6]:
#checking for null values in the lean dataframe after dropping rows with nulls in critical columns

df_lean.isnull().sum()

PRSCRBR_NPI              0
Prscrbr_Last_Org_Name    0
Prscrbr_First_Name       0
Prscrbr_City             0
Prscrbr_State_Abrvtn     0
Tot_Clms                 0
Tot_Drug_Cst             0
dtype: int64

Every doctor must have one row only 
We group by unique NPI and aggregate their metrics


In [7]:
# We use 'first' for demographic data (since it doesn't change) and 'sum' for the metrics

df_hcp_master = df_lean.groupby('PRSCRBR_NPI').agg({
    'Prscrbr_Last_Org_Name': 'first',
    'Prscrbr_First_Name': 'first',
    'Prscrbr_City': 'first',
    'Prscrbr_State_Abrvtn': 'first',
    'Tot_Clms': 'sum',
    'Tot_Drug_Cst': 'sum'
}).reset_index()

In [8]:
#Renaming columns for clarity

df_hcp_master.rename(columns={
    'PRSCRBR_NPI': 'NPI',
    'Prscrbr_Last_Org_Name': 'Last_Name',
    'Prscrbr_First_Name': 'First_Name',
    'Prscrbr_City': 'City',
    'Prscrbr_State_Abrvtn': 'State',
    'Tot_Clms': 'Total_Prescriptions',
    'Tot_Drug_Cst': 'Total_Value'
}, inplace=True)

In [9]:
df_hcp_master

,NPI,Last_Name,First_Name,City,State,Total_Prescriptions,Total_Value
0,1003000936,Stellingworth,Mark,Hartsville,SC,"1,370","$150,258.36"
1,1003006107,Al-Saab,Saad,Tomball,TX,257,"$33,670.94"
2,1003007170,Danciu,Sorin,Chicago,IL,"3,232","$1,149,371.90"
3,1003007204,Yamani,Hussein,Conroe,TX,"6,948","$968,219.45"
4,1003007980,Bernal,Juan,Birmingham,AL,"5,586","$1,236,049.27"
...,...,...,...,...,...,...,...
25215,1992985170,Solomon,Matthew,San Francisco,CA,"1,584","$261,379.22"
25216,1992987770,Turalic,Haris,Lehigh Acres,FL,"11,199","$2,602,559.28"
25217,1992990089,Ala,Chandra,Roseville,MI,152,"$20,829.68"
25218,1992994396,Fram,Ricki,Pine Bluff,AR,"5,698","$545,637.51"


Setting a minimum Threshold for doctors


In [10]:
df_hcp_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 25220 entries, 0 to 25219
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   NPI                  25220 non-null  int64
 1   Last_Name            25220 non-null  str  
 2   First_Name           25220 non-null  str  
 3   City                 25220 non-null  str  
 4   State                25220 non-null  str  
 5   Total_Prescriptions  25220 non-null  str  
 6   Total_Value          25220 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1.3 MB


In [11]:
#converting 'Total_Prescriptions' to numeric type for filtering

#replacing commas in 'Total_Prescriptions' with empty string and converting to float
df_hcp_master['Total_Prescriptions'] = df_hcp_master['Total_Prescriptions'].str.replace(',', '')
df_hcp_master['Total_Prescriptions'] = df_hcp_master['Total_Prescriptions'].astype(float)



In [12]:
#converting 'Total_Prescriptions' to numeric type for filtering
df_final = df_hcp_master[df_hcp_master['Total_Prescriptions'] >= 50]

In [13]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 24252 entries, 0 to 25219
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NPI                  24252 non-null  int64  
 1   Last_Name            24252 non-null  str    
 2   First_Name           24252 non-null  str    
 3   City                 24252 non-null  str    
 4   State                24252 non-null  str    
 5   Total_Prescriptions  24252 non-null  float64
 6   Total_Value          24252 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 1.5 MB


In [14]:
df_final.to_csv('Cardiology_HCP_Master.csv', index=False)